# UC-07 · Evaluate an impeller trim

**Use case:** [UC-07](../docs/product/use-cases.md) - *which impeller diameter makes
the pump meet its rated duty, and what does the trimmed curve look like?*

Method: pure **diameter affinity** (speed unchanged), the same family of laws as the
speed affinity of UC-06:

$$ \frac{Q_2}{Q_1} = \frac{D_2}{D_1},\quad \frac{H_2}{H_1} = \Big(\frac{D_2}{D_1}\Big)^2,\quad
\frac{P_2}{P_1} = \Big(\frac{D_2}{D_1}\Big)^3,\quad \eta_2 = \eta_1 $$

`PerformanceCurve.to_diameter` applies it (and warns beyond 5 % trim);
`pump.trim.diameter_for_duty` intersects the affinity parabola through the duty with
the fitted head curve and returns the diameter.

The curve is **P-9001A** (128 mm impeller, 10 212 rpm, datasheet range 125-131 mm)
corrected to its rated fluid; the duty is the rated 9.80 m3/h at 333.8 m.

**Acceptance:** the needed diameter is about 127.1 mm (-0.7 %); the trimmed curve
passes through the duty; efficiency is unchanged and power scales as D^3.

## 1 · Setup and the P-9001A curve

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

from pump.trim import affinity_intersection, diameter_for_duty

# P-9001A at 10212 rpm, rated fluid (examples/catalogue/P-9001A.pumpcurve.json, rounded)
#       Q [m3/h]  Head [m]  P [kW]  eta [%]
P9001A = [
    (3.4225, 367.746, 18.707, 18.309),
    (6.6198, 365.823, 21.219, 31.058),
    (7.8357, 359.727, 22.492, 34.103),
    (9.7946, 340.692, 24.540, 37.005),
    (10.7853, 321.698, 25.805, 36.589),
    (11.7535, 302.235, 26.923, 35.906),
]
fluid = Fluid("Wash water (rated)", density=Q_(999.0, "kg/m**3"))
speed = Q_(10212, "rpm")
d_ref = Q_(128, "mm")

points = [
    TestPoint(
        fluid=fluid,
        capacity=Q_(q, "m**3/h"),
        speed_of_rotation=speed,
        _head=Q_(h, "m"),
        _efficiency=Q_(e, "percent"),
        breaking_power=Q_(p, "kW"),
    )
    for q, h, p, e in P9001A
]
curve = PerformanceCurve(fluid, points, polynomial_degree=3)

duty_q, duty_h = Q_(9.80, "m**3/h"), Q_(333.8, "m")
print("untrimmed head at the duty flow:", curve.predict_head(duty_q).to("m"))

## 2 · The diameter that meets the duty

The affinity parabola $H = (h/q^2) Q^2$ through the duty meets the head curve at
$Q_1$; then $D = D_{ref}\, q / Q_1$.

In [ ]:
q1 = affinity_intersection(curve, duty_q, duty_h)
d_new = diameter_for_duty(curve, duty_q, duty_h, d_ref)
change = (d_new / d_ref).to("dimensionless").magnitude - 1.0
print(f"Q1 = {q1:.3f~P}   D = {d_new:.2f~P}   ({change:+.2%} vs {d_ref:~P})")
print("inside the datasheet range 125-131 mm:", 125 <= d_new.to("mm").magnitude <= 131)

## 3 · The trimmed curve

`to_diameter` returns a **new** curve; the original is untouched.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    trimmed = curve.to_diameter(d_new, d_ref)
print("warnings for a", f"{change:+.1%}", "trim:", [str(w.message) for w in caught])

ratio = (d_new / d_ref).to("dimensionless").magnitude
print(f"{'Q m3/h':>8} {'H m':>8} {'P kW':>7} {'eta %':>6}")
for p in trimmed.points:
    print(
        f"{p.capacity.magnitude:8.3f} {p.head.magnitude:8.2f} "
        f"{p.breaking_power.magnitude:7.2f} {p.efficiency.magnitude:6.2f}"
    )

fig, ax = plt.subplots(figsize=(7, 4.5))
for c, label in ((curve, "128 mm"), (trimmed, f"{d_new.magnitude:.1f} mm")):
    caps = np.asarray(c.fitter.capacities, dtype=float)
    xs = np.linspace(caps.min(), caps.max(), 100)
    ax.plot(xs, np.polyval(c.fitter.head_coeffs, xs), label=label)
ax.plot(duty_q.magnitude, duty_h.magnitude, "k*", ms=14, label="rated duty")
ax.set_xlabel("Capacity (m3/h)")
ax.set_ylabel("Head (m)")
ax.grid(alpha=0.3)
ax.legend(title="Impeller")
fig.tight_layout()
plt.show()

## 4 · Limits of the method

A trim beyond `max_trim` (5 % by default) warns that the approximation loses accuracy;
a duty that the curve cannot reach by trimming raises `ValueError`.

In [ ]:
with warnings.catch_warnings(record=True) as caught_big:
    warnings.simplefilter("always")
    big = curve.to_diameter(Q_(110, "mm"), d_ref)
big_warned = any(issubclass(w.category, UserWarning) for w in caught_big)

with warnings.catch_warnings(record=True) as caught_ok:
    warnings.simplefilter("always")
    curve.to_diameter(Q_(110, "mm"), d_ref, max_trim=0.2)
big_ok = not caught_ok

try:
    diameter_for_duty(curve, duty_q, Q_(50, "m"), d_ref)
    unreachable = False
except ValueError as exc:
    unreachable = True
    print("ValueError:", str(exc)[:110], "...")

## 5 · Acceptance

In [ ]:
# diameter (regression value of the project: 127.1 mm, -0.7 %)
assert 126.9 < d_new.to("mm").magnitude < 127.3
assert -0.012 < change < -0.003
assert abs(d_new.to("mm").magnitude - 128 * 9.80 / q1.magnitude) < 1e-9

# no warning for a small trim; the original curve is not mutated
assert not caught
assert curve.points[0].capacity.magnitude == P9001A[0][0]

# the trimmed curve passes through the duty
assert abs(trimmed.predict_head(duty_q).to("m").magnitude - 333.8) < 1e-3

# affinity laws, point by point; eta unchanged
for old, new in zip(curve.points, trimmed.points):
    assert abs(new.capacity.magnitude - old.capacity.magnitude * ratio) < 1e-9
    assert abs(new.head.magnitude - old.head.magnitude * ratio**2) < 1e-9
    assert abs(new.breaking_power.magnitude - old.breaking_power.magnitude * ratio**3) < 1e-9
    assert abs(new.efficiency.magnitude - old.efficiency.magnitude) < 1e-12

# limits
assert big_warned and big_ok and unreachable
print("UC-07 acceptance OK")